In [1]:
import pandas as pd
import numpy as np
import re
import os

print("Libraries loaded ✓")
print("Pandas version:", pd.__version__)

Libraries loaded ✓
Pandas version: 2.3.0


In [5]:

MOCK_PATH = r"C:\Users\Raksha\OneDrive\Desktop\KCET-Recommendation-System\data\mockAllotmentCutoff_2025.csv"
ROUND1_PATH = r"C:\Users\Raksha\OneDrive\Desktop\KCET-Recommendation-System\data\round1Cutoff2025.csv"
ROUND2_PATH = r"C:\Users\Raksha\OneDrive\Desktop\KCET-Recommendation-System\data\round2Cutoff2025.csv"
ROUND3_PATH = r"C:\Users\Raksha\OneDrive\Desktop\KCET-Recommendation-System\data\round3Cutoff_2025.csv"

for name, path in [('Mock', MOCK_PATH), ('Round1', ROUND1_PATH), ('Round2', ROUND2_PATH), ('Round3', ROUND3_PATH)]:
    exists = os.path.exists(path)
    print(f'{name}: {"✓ Found" if exists else "✗ NOT FOUND — check your file path!"}')

Mock: ✓ Found
Round1: ✓ Found
Round2: ✓ Found
Round3: ✓ Found


In [6]:
# Load all 4 files and peek at each one
datasets = {
    'Mock'  : MOCK_PATH,
    'Round1': ROUND1_PATH,
    'Round2': ROUND2_PATH,
    'Round3': ROUND3_PATH,
}

for name, path in datasets.items():
    df = pd.read_csv(path, on_bad_lines='skip', header=None)
    print(f"=== {name} ===")
    print(f"  Rows: {df.shape[0]}  |  Columns: {df.shape[1]}")
    print(f"  First cell: {str(df.iloc[0,0])[:60]}")
    print()

=== Mock ===
  Rows: 5470  |  Columns: 25
  First cell: nan

=== Round1 ===
  Rows: 6540  |  Columns: 1
  First cell: KARNATAKA EXAMINATIONS AUTHORITY

=== Round2 ===
  Rows: 5877  |  Columns: 29
  First cell: nan

=== Round3 ===
  Rows: 7032  |  Columns: 1
  First cell: KARNATAKA EXAMINATIONS AUTHORITY



In [7]:
# Peek inside Mock Allotment file
df_mock = pd.read_csv(MOCK_PATH, on_bad_lines='skip', header=None)

print("=== MOCK — First 6 rows ===")
print(df_mock.head(6).to_string())

print("\n=== MOCK — Row where headers appear ===")
for i, row in df_mock.iterrows():
    if 'Course Name' in str(row[0]) or 'Course Name' in str(row[1]):
        print(f"Header at row {i}:")
        print(row.tolist())
        break

=== MOCK — First 6 rows ===
                                                                                                                                          0    1    2    3    4                                                                                       5                                                        6    7                                 8    9    10   11   12   13   14   15   16   17   18   19   20   21   22   23   24
0                                                                                                                                        NaN  NaN  NaN  NaN  NaN                                                                                     NaN                                                      NaN  NaN  KARNATAKA EXAMINATIONS AUTHORITY  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN
1                                                                                                                         

In [8]:
# ── PARSER FOR CSV FORMAT (Mock + Round2) ──────────────────────
def parse_csv_format(path, round_name):
    
    cat_cols_24 = ['1G','1K','1R','2AG','2AK','2AR','2BG','2BK','2BR',
                   '3AG','3AK','3AR','3BG','3BK','3BR','GM','GMK','GMR',
                   'SCG','SCK','SCR','STG','STK','STR']
    
    cat_cols_28 = ['1G','1K','1R','2AG','2AK','2AR','2BG','2BK','2BR',
                   '3AG','3AK','3AR','3BG','3BK','3BR','GM','GMK','GMP',
                   'GMR','NRI','OPN','OTH','SCG','SCK','SCR','STG','STK','STR']

    records = []
    current_college_code = None
    current_college_name = None
    current_branch_parts = []
    current_branch_values = None

    def save_branch():
        if current_branch_parts and current_branch_values is not None:
            branch_name = " ".join(current_branch_parts)
            # detect which cat_cols to use based on number of value columns
            cat_cols = cat_cols_28 if len(current_branch_values) > 25 else cat_cols_24
            rec = {
                'college_code': current_college_code,
                'college_name': current_college_name,
                'branch': branch_name,
                'round': round_name
            }
            for i, col in enumerate(cat_cols):
                val = str(current_branch_values[i+1]).strip() if i+1 < len(current_branch_values) else '--'
                rec[col] = None if val in ['--', 'nan', 'NaN', ''] else val
            records.append(rec)

    df_raw = pd.read_csv(path, on_bad_lines='skip', header=None)
    
    for idx, row in df_raw.iterrows():
        cell0 = str(row[0]).strip() if pd.notna(row[0]) else ''
        
        if 'College:' in cell0:
            save_branch()
            current_branch_parts = []
            current_branch_values = None
            college_part = cell0.split('College:')[1].strip()
            match = re.match(r'\(?([A-Z]\d{3})\)?(.+)', college_part)
            if match:
                current_college_code = match.group(1)
                current_college_name = match.group(2).strip()
            continue
        
        if cell0 == 'Course Name' or 'Generated on' in cell0 or cell0 in ['nan', '']:
            continue
        
        has_values = any(str(row[i]).strip() not in ['nan', '', 'NaN'] for i in range(1, 10))
        
        if has_values:
            save_branch()
            current_branch_parts = [cell0] if cell0 else []
            current_branch_values = row
        else:
            if cell0 and cell0 not in ['nan', '']:
                current_branch_parts.append(cell0)

    save_branch()
    return pd.DataFrame(records)


# ── PARSER FOR SPACE-DELIMITED FORMAT (Round1 + Round3) ────────
def parse_space_delimited(path, round_name):

    cat_cols_24 = ['1G','1K','1R','2AG','2AK','2AR','2BG','2BK','2BR',
                   '3AG','3AK','3AR','3BG','3BK','3BR','GM','GMK','GMR',
                   'SCG','SCK','SCR','STG','STK','STR']
    
    cat_cols_28 = ['1G','1K','1R','2AG','2AK','2AR','2BG','2BK','2BR',
                   '3AG','3AK','3AR','3BG','3BK','3BR','GM','GMK','GMP',
                   'GMR','NRI','OPN','OTH','SCG','SCK','SCR','STG','STK','STR']

    records = []
    current_college_code = None
    current_college_name = None
    current_branch_parts = []
    current_branch_values = None

    def save_branch():
        if current_branch_parts and current_branch_values is not None:
            branch_name = " ".join(current_branch_parts)
            cat_cols = cat_cols_28 if len(current_branch_values) > 25 else cat_cols_24
            rec = {
                'college_code': current_college_code,
                'college_name': current_college_name,
                'branch': branch_name,
                'round': round_name
            }
            for i, col in enumerate(cat_cols):
                val = current_branch_values[i] if i < len(current_branch_values) else '--'
                rec[col] = None if val in ['--', '', 'nan'] else val
            records.append(rec)

    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            line = line.strip().strip('"')
            if not line:
                continue
            if 'College:' in line:
                save_branch()
                current_branch_parts = []
                current_branch_values = None
                college_part = line.split('College:')[1].strip()
                match = re.match(r'\(?([A-Z]\d{3})\)?(.+)', college_part)
                if match:
                    current_college_code = match.group(1)
                    current_college_name = match.group(2).strip()
                continue
            if line.startswith('Course Name') or 'Generated on' in line:
                continue
            tokens = re.split(r'\s{2,}', line)
            has_numbers = any(re.match(r'^\d', t.strip()) or t.strip() == '--' for t in tokens[1:])
            if has_numbers and len(tokens) > 1:
                save_branch()
                current_branch_parts = [tokens[0].strip()] if tokens[0].strip() else []
                current_branch_values = [t.strip() for t in tokens[1:]]
            else:
                if line:
                    current_branch_parts.append(line.strip())

    save_branch()
    return pd.DataFrame(records)


print("Parser functions defined ✓")

Parser functions defined ✓


In [9]:
# Parse all 4 files
print("Parsing Mock Allotment...")
df_mock = parse_csv_format(MOCK_PATH, 'Mock')
print(f"  → {len(df_mock)} rows | {df_mock['college_code'].nunique()} colleges")

print("Parsing Round 1...")
df_r1 = parse_space_delimited(ROUND1_PATH, 'Round1')
print(f"  → {len(df_r1)} rows | {df_r1['college_code'].nunique()} colleges")

print("Parsing Round 2...")
df_r2 = parse_csv_format(ROUND2_PATH, 'Round2')
print(f"  → {len(df_r2)} rows | {df_r2['college_code'].nunique()} colleges")

print("Parsing Round 3...")
df_r3 = parse_space_delimited(ROUND3_PATH, 'Round3')
print(f"  → {len(df_r3)} rows | {df_r3['college_code'].nunique()} colleges")

print("\nAll files parsed ✓")

Parsing Mock Allotment...
  → 1567 rows | 221 colleges
Parsing Round 1...
  → 1610 rows | 227 colleges
Parsing Round 2...
  → 1602 rows | 221 colleges
Parsing Round 3...
  → 1821 rows | 229 colleges

All files parsed ✓


In [10]:
# Merge all 4 into one dataframe
df_combined = pd.concat([df_mock, df_r1, df_r2, df_r3], ignore_index=True)

print(f"Combined shape: {df_combined.shape}")
print(f"\nRows per round:")
print(df_combined['round'].value_counts().to_string())
print(f"\nTotal colleges: {df_combined['college_code'].nunique()}")
print(f"Total branches: {df_combined['branch'].nunique()}")
print(f"\nFirst 3 rows:")
df_combined[['college_code', 'branch', 'round', 'GM', 'SCG']].head(3)

Combined shape: (6600, 32)

Rows per round:
round
Round3    1821
Round1    1610
Round2    1602
Mock      1567

Total colleges: 229
Total branches: 262

First 3 rows:


,college_code,branch,round,GM,SCG
0,E001,ARTIFICIAL INTELLIGENCE AND DATA SCIENCE,Mock,4550,26335
1,E001,CIVIL ENGINEERING,Mock,42449,68702
2,E001,COMPUTER SCIENCE AND ENGINEERING,Mock,3473,21814


In [11]:
# Fix broken branch names
BRANCH_RENAME = {
    'ELECTRONICS AND COMMUNICATIO N ENGG'                          : 'ELECTRONICS AND COMMUNICATION ENGG',
    'B TECH IN ELECTRONICS & COMMUNICATIO N ENGINEERING'           : 'B TECH IN ELECTRONICS & COMMUNICATION ENGINEERING',
    'COMPUTER SCIENCE AND ENGG(ARTIFICIA L INTELLIGENCE AND MACHINE LEARNING)' : 'COMPUTER SCIENCE AND ENGG (ARTIFICIAL INTELLIGENCE AND MACHINE LEARNING)',
    'COMPUTER SCIENCE AND ENGINEERING(D ATA SCIENCE)'              : 'COMPUTER SCIENCE AND ENGINEERING (DATA SCIENCE)',
    'COMPUTER SCIENCE AND ENGG(INTERNE T OF THINGS & CYBER SECURITY INCLUDING BLOCK CHAIN TECH)' : 'COMPUTER SCIENCE AND ENGG (INTERNET OF THINGS & CYBER SECURITY INCLUDING BLOCK CHAIN TECH)',
    'COMPUTER SCIENCE AND ENGG. (ARTIF ICIAL INTELLIGENCE)'        : 'COMPUTER SCIENCE AND ENGG. (ARTIFICIAL INTELLIGENCE)',
    'ELECTRONICS AND TELECOMMUNICATIO N ENGINEERING'               : 'ELECTRONICS AND TELECOMMUNICATION ENGINEERING',
}

before = df_combined['branch'].nunique()
df_combined['branch'] = df_combined['branch'].replace(BRANCH_RENAME)
after = df_combined['branch'].nunique()

print(f"Branch names before fix: {before}")
print(f"Branch names after fix:  {after}")
print(f"Duplicates removed:      {before - after}")

Branch names before fix: 262
Branch names after fix:  260
Duplicates removed:      2


In [12]:
# Find all branch names that look broken (have a space mid-word)
# Broken names have patterns like "COMMUNICATIO N" or "ARTIFICIA L"
all_branches = df_combined['branch'].unique()

print("Potentially broken branch names:")
print("(looking for single letters or 2-letter chunks split by spaces)\n")

broken = []
for branch in sorted(all_branches):
    words = branch.split(' ')
    for i, word in enumerate(words):
        # A word with 1-3 letters surrounded by longer words is suspicious
        if len(word) <= 3 and i > 0 and i < len(words)-1:
            if len(words[i-1]) > 3:  # previous word is long
                broken.append(branch)
                break

for b in broken:
    print(f"  '{b}'")

print(f"\nTotal suspicious: {len(broken)}")

Potentially broken branch names:
(looking for single letters or 2-letter chunks split by spaces)

  'AND COMMUNICATIO N ENGG'
  'AND INSTRUMENTATI ON ENGINEERING'
  'ARTIFICIAL INTELLIGENCE AND DATA SCIENCE'
  'ARTIFICIAL INTELLIGENCE AND MACHINE LEARNING'
  'AUTOMATION AND ROBOTICS'
  'B TECH (HONS) COMPUTER SCIENCE AND ENGINEERING(DAT A SCIENCE)'
  'B TECH IN AERO SPACE ENGINEERING'
  'B TECH IN AERONAUTICAL ENGINEERING'
  'B TECH IN AGRICULTURAL ENGINEERING'
  'B TECH IN ARTIFICIAL INTELLIGENCE AND DATA SCIENCE'
  'B TECH IN ARTIFICIAL INTELLIGENCE AND MACHINE LEARNING'
  'B TECH IN BIO-'
  'B TECH IN BIO- TECHNOLOGY'
  'B TECH IN CIVIL ENGINEERING'
  'B TECH IN COMPUTER'
  'B TECH IN COMPUTER ENGINEERING'
  'B TECH IN COMPUTER SCIENCE'
  'B TECH IN COMPUTER SCIENCE & ENGG (ARTIFICIAL INTELLIGENCE AND FUTURE TECHNOLOGIES )'
  'B TECH IN COMPUTER SCIENCE & ENGG (ARTIFICIAL INTELLIGENCE AND FUTURE TECHNOLOGIES)'
  'B TECH IN COMPUTER SCIENCE & ENGINEERING (ARTIFICAL INTELLIGENCE & MAC

In [13]:
# Complete branch name fix map
BRANCH_RENAME = {
    # Electronics & Communication splits
    'AND COMMUNICATIO N ENGG'                          : 'ELECTRONICS AND COMMUNICATION ENGG',
    'COMMUNICATIO N DESIGN'                            : 'COMMUNICATION DESIGN',
    'COMPUTER AND COMMUNICATIO N ENGINEERING'          : 'COMPUTER AND COMMUNICATION ENGINEERING',
    'ELECTRONICS & COMMUNICATIO N ENGINEERING'         : 'ELECTRONICS & COMMUNICATION ENGINEERING',
    'ELECTRONICS & COMMUNICATIO N ENGINEERING(I NDUSTRIAL INTEGTATED)' : 'ELECTRONICS & COMMUNICATION ENGINEERING (INDUSTRIAL INTEGRATED)',
    'ELECTRONICS AND COMMUNICATIO N ENGG (VLSI DESIGN AND TECHNOLOGY)' : 'ELECTRONICS AND COMMUNICATION ENGG (VLSI DESIGN AND TECHNOLOGY)',
    'ELECTRONICS AND COMMUNICATIO N (ADVANCED COMMUNICATIO N TECHNOLOGY)' : 'ELECTRONICS AND COMMUNICATION (ADVANCED COMMUNICATION TECHNOLOGY)',
    'AND INSTRUMENTATI ON ENGINEERING'                 : 'ELECTRONICS AND INSTRUMENTATION ENGINEERING',
    'ELECTRONICS & INSTRUMENTATI ON ENGINEERING'       : 'ELECTRONICS & INSTRUMENTATION ENGINEERING',
    'ELECTRONICS AND INSTRUMENTATI ON ENGINEERING'     : 'ELECTRONICS AND INSTRUMENTATION ENGINEERING',
    'ELECTRONICS AND TELECOMMUNIC ATION ENGINEERING'   : 'ELECTRONICS AND TELECOMMUNICATION ENGINEERING',
    'ELECTRONICS AND TELECOMMUNICAT ION ENGINEERING'   : 'ELECTRONICS AND TELECOMMUNICATION ENGINEERING',

    # Computer Science splits
    'COMPUTER SCIENCE AND ENGG(INTERNE T OF THINGS)'  : 'COMPUTER SCIENCE AND ENGG (INTERNET OF THINGS)',
    'COMPUTER SCIENCE AND ENGINEERING(A RTIFICAL INTELLIGENCE & DATA SCIENCE)' : 'COMPUTER SCIENCE AND ENGINEERING (ARTIFICIAL INTELLIGENCE & DATA SCIENCE)',
    'COMPUTER SCIENCE AND ENGINEERING(A RTIFICIAL INTELLIGENCE AND DATA SCIENCE)' : 'COMPUTER SCIENCE AND ENGINEERING (ARTIFICIAL INTELLIGENCE AND DATA SCIENCE)',
    'COMPUTER SCIENCE AND ENGINEERING(ARTI FICAL INTELLIGENCE & DATA SCIENCE)' : 'COMPUTER SCIENCE AND ENGINEERING (ARTIFICIAL INTELLIGENCE & DATA SCIENCE)',
    'COMPUTER SCIENCE AND ENGINEERING(C YBER SECURITY)' : 'COMPUTER SCIENCE AND ENGINEERING (CYBER SECURITY)',
    'COMPUTER SCIENCE AND ENGINEERING(DAT A SCIENCE)'  : 'COMPUTER SCIENCE AND ENGINEERING (DATA SCIENCE)',
    'COMPUTER SCIENCE AND ENGINEERING(I OT)'           : 'COMPUTER SCIENCE AND ENGINEERING (IOT)',
    'COMPUTER SCIENCE AND TECHNOLOGY(B IG DATA)'       : 'COMPUTER SCIENCE AND TECHNOLOGY (BIG DATA)',
    'COMPUTER SCIENCE AND TECHNOLOGY(E XCLUSIVELY FOR DIFFERENTLY ABLED)' : 'COMPUTER SCIENCE AND TECHNOLOGY (EXCLUSIVELY FOR DIFFERENTLY ABLED)',
    'COMPUTER SCIENCE AND TECHNOLOGY(EXC LUSIVELY FOR DIFFERENTLY ABLED)' : 'COMPUTER SCIENCE AND TECHNOLOGY (EXCLUSIVELY FOR DIFFERENTLY ABLED)',
    'Computer Science and Engineering(Clou d Computing)' : 'COMPUTER SCIENCE AND ENGINEERING (CLOUD COMPUTING)',

    # B TECH splits
    'B TECH IN COMPUTER SCIENCE AND ENGINEERING(A RTIFICIAL INTELLIGENCE AND DATA SCIENCE)' : 'B TECH IN COMPUTER SCIENCE AND ENGINEERING (ARTIFICIAL INTELLIGENCE AND DATA SCIENCE)',
    'B TECH IN COMPUTER SCIENCE AND ENGINEERING(ARTI FICIAL INTELLIGENCE AND DATA SCIENCE)' : 'B TECH IN COMPUTER SCIENCE AND ENGINEERING (ARTIFICIAL INTELLIGENCE AND DATA SCIENCE)',
    'B TECH IN COMPUTER SCIENCE AND ENGINEERING(B LOCK CHAIN)' : 'B TECH IN COMPUTER SCIENCE AND ENGINEERING (BLOCK CHAIN)',
    'B TECH IN COMPUTER SCIENCE AND ENGINEERING(BLO CK CHAIN)' : 'B TECH IN COMPUTER SCIENCE AND ENGINEERING (BLOCK CHAIN)',
    'B TECH IN COMPUTER SCIENCE AND ENGINEERING(C YBER SECURITY)' : 'B TECH IN COMPUTER SCIENCE AND ENGINEERING (CYBER SECURITY)',
    'B TECH IN COMPUTER SCIENCE AND ENGINEERING(CYB ER SECURITY)' : 'B TECH IN COMPUTER SCIENCE AND ENGINEERING (CYBER SECURITY)',
    'B TECH IN COMPUTER SCIENCE AND ENGINEERING(D ATA SCIENCE)' : 'B TECH IN COMPUTER SCIENCE AND ENGINEERING (DATA SCIENCE)',
    'B TECH IN COMPUTER SCIENCE AND ENGINEERING(DAT A SCIENCE)' : 'B TECH IN COMPUTER SCIENCE AND ENGINEERING (DATA SCIENCE)',
    'B TECH IN COMPUTER SCIENCE AND ENGINEERING(I OT INCLUDING BLOCK CHAIN)' : 'B TECH IN COMPUTER SCIENCE AND ENGINEERING (IOT INCLUDING BLOCK CHAIN)',
    'B TECH IN COMPUTER SCIENCE AND TECHNOLOGY(B IG DATA)' : 'B TECH IN COMPUTER SCIENCE AND TECHNOLOGY (BIG DATA)',
    'B TECH IN COMPUTER SCIENCE AND TECHNOLOGY(D EV OPS)' : 'B TECH IN COMPUTER SCIENCE AND TECHNOLOGY (DEV OPS)',
    'B TECH IN COMPUTER SCIENCE AND TECHNOLOGY(DEV OPS)' : 'B TECH IN COMPUTER SCIENCE AND TECHNOLOGY (DEV OPS)',
    'B TECH IN BIO-'                                   : 'B TECH IN BIO-TECHNOLOGY',
    'B TECH IN BIO- TECHNOLOGY'                        : 'B TECH IN BIO-TECHNOLOGY',
    'B.TECH IN COMPUTER ENGINEERING(S OFTWARE PRODUCT DEVELOPMENT)' : 'B.TECH IN COMPUTER ENGINEERING (SOFTWARE PRODUCT DEVELOPMENT)',
    'B.TECH IN COMPUTER ENGINEERING(SOF TWARE PRODUCT DEVELOPMENT)' : 'B.TECH IN COMPUTER ENGINEERING (SOFTWARE PRODUCT DEVELOPMENT)',
    'B.Tech In BIOTECHNOLOG Y & BIO- ENGINEERING'      : 'B.TECH IN BIOTECHNOLOGY & BIO-ENGINEERING',
    'B.Tech In BIOTECHNOLOGY & BIO- ENGINEERING'       : 'B.TECH IN BIOTECHNOLOGY & BIO-ENGINEERING',
    'BTECH IN ELECTRONICS ENGINEERING(V LSI DESIGN & TECHNOLOGY)' : 'BTECH IN ELECTRONICS ENGINEERING (VLSI DESIGN & TECHNOLOGY)',
    'B TECH IN AERO SPACE ENGINEERING'                 : 'B TECH IN AEROSPACE ENGINEERING',
    'B TECH (HONS) COMPUTER SCIENCE AND ENGINEERING(DAT A SCIENCE)' : 'B TECH (HONS) COMPUTER SCIENCE AND ENGINEERING (DATA SCIENCE)',

    # Other splits
    'CIVIL ENVIRONMENTA L ENGINEERING'                 : 'CIVIL ENVIRONMENTAL ENGINEERING',
    'ENVIRONMENTA L ENGINEERING'                       : 'ENVIRONMENTAL ENGINEERING',
    'ELECTRONICS ENGINEERING(V LSI DESIGN & TECHNOLOGY)' : 'ELECTRONICS ENGINEERING (VLSI DESIGN & TECHNOLOGY)',
    'ENGINEERING(V LSI DESIGN & TECHNOLOGY)'           : 'ELECTRONICS ENGINEERING (VLSI DESIGN & TECHNOLOGY)',
    'MECHANICAL AND SMART MANUFACTURIN G'              : 'MECHANICAL AND SMART MANUFACTURING',
    'BTECH IN MECHANICAL AND SMART MANUFACTURIN G'     : 'BTECH IN MECHANICAL AND SMART MANUFACTURING',
    'BTECH IN PHARMACEUTIC AL ENGINEERING'             : 'BTECH IN PHARMACEUTICAL ENGINEERING',

    # Fragment names (branch name got cut — prefix missing)
    'INTELLIGENCE AND DATA SCIENCE'                    : 'ARTIFICIAL INTELLIGENCE AND DATA SCIENCE',
    'INTELLIGENCE AND MACHINE LEARNING'                : 'ARTIFICIAL INTELLIGENCE AND MACHINE LEARNING',
    'SCIENCE AND BUSINESS SYSTEMS'                     : 'COMPUTER SCIENCE AND BUSINESS SYSTEMS',
    'SCIENCE AND DESIGN'                               : 'COMPUTER SCIENCE AND DESIGN',
    'SCIENCE AND ENGINEERING'                          : 'COMPUTER SCIENCE AND ENGINEERING',
    'SCIENCE AND ENGINEERING (AIML)'                   : 'COMPUTER SCIENCE AND ENGINEERING (AIML)',
    'SCIENCE AND ENGINEERING (CYBER SECURITY)'         : 'COMPUTER SCIENCE AND ENGINEERING (CYBER SECURITY)',
    'SCIENCE AND ENGINEERING(A RTIFICAL INTELLIGENCE & DATA SCIENCE)' : 'COMPUTER SCIENCE AND ENGINEERING (ARTIFICIAL INTELLIGENCE & DATA SCIENCE)',
    'SCIENCE AND ENGINEERING(D ATA SCIENCE)'           : 'COMPUTER SCIENCE AND ENGINEERING (DATA SCIENCE)',
    'SCIENCE AND ENGINEERING(DAT A SCIENCE)'           : 'COMPUTER SCIENCE AND ENGINEERING (DATA SCIENCE)',
    'SCIENCE AND ENGG (ARTIFICIAL INTELLIGENCE)'       : 'COMPUTER SCIENCE AND ENGG (ARTIFICIAL INTELLIGENCE)',
    'SCIENCE AND ENGG(ARTIFICIA L INTELLIGENCE AND MACHINE LEARNING)' : 'COMPUTER SCIENCE AND ENGG (ARTIFICIAL INTELLIGENCE AND MACHINE LEARNING)',
    'SCIENCE AND ENGG(ARTIFICIAL INTELLIGENCE AND MACHINE LEARNING)' : 'COMPUTER SCIENCE AND ENGG (ARTIFICIAL INTELLIGENCE AND MACHINE LEARNING)',
    'SCIENCE AND ENGG(INTERNE T OF THINGS & CYBER SECURITY INCLUDING BLOCK CHAIN TECH)' : 'COMPUTER SCIENCE AND ENGG (INTERNET OF THINGS & CYBER SECURITY INCLUDING BLOCK CHAIN TECH)',
    'SCIENCE AND ENGG(INTERNET OF THINGS & CYBER SECURITY INCLUDING BLOCK CHAIN TECH)' : 'COMPUTER SCIENCE AND ENGG (INTERNET OF THINGS & CYBER SECURITY INCLUDING BLOCK CHAIN TECH)',
    'TECHNOLOGY AND MGMT'                              : 'ENGINEERING & MANAGEMENT',

    # Trailing spaces
    'ELECTRICAL & ELECTRONICS ENGINEERING '            : 'ELECTRICAL & ELECTRONICS ENGINEERING',
    'INFORMATION SCIENCE AND ENGINEERING '             : 'INFORMATION SCIENCE AND ENGINEERING',
}

before = df_combined['branch'].nunique()
df_combined['branch'] = df_combined['branch'].replace(BRANCH_RENAME)
# Also strip trailing/leading spaces from all branch names
df_combined['branch'] = df_combined['branch'].str.strip()
after = df_combined['branch'].nunique()

print(f"Branch names before: {before}")
print(f"Branch names after:  {after}")
print(f"Reduced by:          {before - after}")

Branch names before: 260
Branch names after:  215
Reduced by:          45


In [14]:
# Drop columns that are 100% empty
cat_cols = ['1G','1K','1R','2AG','2AK','2AR','2BG','2BK','2BR',
            '3AG','3AK','3AR','3BG','3BK','3BR','GM','GMK','GMP',
            'GMR','NRI','OPN','OTH','SCG','SCK','SCR','STG','STK','STR']

# Only work with columns that actually exist in our dataframe
cat_cols_present = [c for c in cat_cols if c in df_combined.columns]

empty_cols = [c for c in cat_cols_present if df_combined[c].isna().all()]
print(f"Dropping 100% empty columns: {empty_cols}")

df_combined.drop(columns=empty_cols, inplace=True)
cat_cols_present = [c for c in cat_cols_present if c not in empty_cols]
print(f"Remaining category columns: {len(cat_cols_present)}")

Dropping 100% empty columns: []
Remaining category columns: 28


In [15]:
# Melt — wide to long format
id_cols = ['college_code', 'college_name', 'branch', 'round']
cat_cols_present = [c for c in df_combined.columns if c not in id_cols]

df_long = df_combined.melt(
    id_vars    = id_cols,
    value_vars = cat_cols_present,
    var_name   = 'category',
    value_name = 'cutoff_rank'
)

# Drop rows with no cutoff data
df_long = df_long.dropna(subset=['cutoff_rank'])

# Convert to number
df_long['cutoff_rank'] = pd.to_numeric(df_long['cutoff_rank'], errors='coerce')
df_long = df_long.dropna(subset=['cutoff_rank'])

# Convert to integer
df_long['cutoff_rank'] = df_long['cutoff_rank'].astype(int)

# Sort nicely
df_long = df_long.sort_values(
    ['round', 'college_code', 'branch', 'category']
).reset_index(drop=True)

print(f"Long format shape: {df_long.shape}")
print(f"\nRows per round:")
print(df_long['round'].value_counts().to_string())
print(f"\nSample rows:")
df_long.head(8)

Long format shape: (62686, 6)

Rows per round:
round
Round1    18894
Mock      18420
Round2    14317
Round3    11055

Sample rows:


,college_code,college_name,branch,round,category,cutoff_rank
0,E001,Univesity of Visvesvaraya College of Engineeri...,ARTIFICIAL INTELLIGENCE AND DATA SCIENCE,Mock,1G,7516
1,E001,Univesity of Visvesvaraya College of Engineeri...,ARTIFICIAL INTELLIGENCE AND DATA SCIENCE,Mock,2AG,6740
2,E001,Univesity of Visvesvaraya College of Engineeri...,ARTIFICIAL INTELLIGENCE AND DATA SCIENCE,Mock,2AR,7574
3,E001,Univesity of Visvesvaraya College of Engineeri...,ARTIFICIAL INTELLIGENCE AND DATA SCIENCE,Mock,2BG,10625
4,E001,Univesity of Visvesvaraya College of Engineeri...,ARTIFICIAL INTELLIGENCE AND DATA SCIENCE,Mock,3AG,5191
5,E001,Univesity of Visvesvaraya College of Engineeri...,ARTIFICIAL INTELLIGENCE AND DATA SCIENCE,Mock,3BG,4814
6,E001,Univesity of Visvesvaraya College of Engineeri...,ARTIFICIAL INTELLIGENCE AND DATA SCIENCE,Mock,GM,4550
7,E001,Univesity of Visvesvaraya College of Engineeri...,ARTIFICIAL INTELLIGENCE AND DATA SCIENCE,Mock,GMK,9379


In [16]:
import os

# Make sure data folder exists
os.makedirs('../data', exist_ok=True)

# Save 1 — All rounds combined (for ML later)
df_long.to_csv('../data/cutoffs_2025_all_rounds.csv', index=False)
print(f"Saved: cutoffs_2025_all_rounds.csv  ({len(df_long):,} rows)")

# Save 2 — Round 3 only (for recommendation engine V1)
df_round3 = df_long[df_long['round'] == 'Round3'].copy()
df_round3.to_csv('../data/cutoffs_2025_clean.csv', index=False)
print(f"Saved: cutoffs_2025_clean.csv        ({len(df_round3):,} rows)")

print("\nDone! ✓")
print(f"\ncutoffs_2025_clean.csv summary:")
print(f"  Colleges : {df_round3['college_code'].nunique()}")
print(f"  Branches : {df_round3['branch'].nunique()}")
print(f"  Categories: {df_round3['category'].nunique()}")

Saved: cutoffs_2025_all_rounds.csv  (62,686 rows)
Saved: cutoffs_2025_clean.csv        (11,055 rows)

Done! ✓

cutoffs_2025_clean.csv summary:
  Colleges : 229
  Branches : 173
  Categories: 27
